# Gold Layer Modeling 

**Project:** Marketing Data Pipeline – Logicrest  
**Layer:** Gold Layer  
**Model:** Star Schema + KPI Output Tables

## Objective
Transform the validated Silver dataset into a business-ready Gold layer using a PDF-aligned warehouse model.

### Gold Layer Components
**Dimensions**
- dim_date
- dim_product
- dim_geo
- dim_campaign
- dim_seller_channel

**Fact**
- fact_marketing_daily

**KPI Output Tables**
- kpi_brand_performance
- kpi_campaign_performance
- kpi_platform_performance
- kpi_geo_performance
- kpi_product_tier_performance
- kpi_water_type_performance
- kpi_monthly_performance

The Gold layer is designed for downstream consumption in:
**SSMS → Power BI**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "water_bottle_db"

SILVER_TBL = f"{CATALOG}.{SCHEMA}.silver_packaged_water_marketing_clean"

DIM_DATE_TBL           = f"{CATALOG}.{SCHEMA}.dim_date"
DIM_PRODUCT_TBL        = f"{CATALOG}.{SCHEMA}.dim_product"
DIM_GEO_TBL            = f"{CATALOG}.{SCHEMA}.dim_geo"
DIM_CAMPAIGN_TBL       = f"{CATALOG}.{SCHEMA}.dim_campaign"
DIM_SELLER_CHANNEL_TBL = f"{CATALOG}.{SCHEMA}.dim_seller_channel"
FACT_TBL               = f"{CATALOG}.{SCHEMA}.fact_marketing_daily"

KPI_BRAND_TBL          = f"{CATALOG}.{SCHEMA}.kpi_brand_performance"
KPI_CAMPAIGN_TBL       = f"{CATALOG}.{SCHEMA}.kpi_campaign_performance"
KPI_PLATFORM_TBL       = f"{CATALOG}.{SCHEMA}.kpi_platform_performance"
KPI_GEO_TBL            = f"{CATALOG}.{SCHEMA}.kpi_geo_performance"
KPI_TIER_TBL           = f"{CATALOG}.{SCHEMA}.kpi_product_tier_performance"
KPI_WATER_TBL          = f"{CATALOG}.{SCHEMA}.kpi_water_type_performance"
KPI_MONTHLY_TBL        = f"{CATALOG}.{SCHEMA}.kpi_monthly_performance"

FULL_LOAD = False

def norm_upper(col_name):
    return F.upper(F.trim(F.regexp_replace(F.col(col_name), r"\s+", " ")))

def sha256_key(*cols):
    return F.sha2(F.concat_ws("|", *cols), 256)

SIZE_TOKEN_RE = r"(?i)\b\d+(\.\d+)?\s*(ML|L)\b"

print("Gold setup complete.")
print("Silver source:", SILVER_TBL)
print("Fact target  :", FACT_TBL)

Gold setup complete.
Silver source: workspace.water_bottle_db.silver_packaged_water_marketing_clean
Fact target  : workspace.water_bottle_db.fact_marketing_daily


Interpretation: Central setup for table names, runtime flags, helper functions, and notebook-wide constants.

In [0]:
silver_df = spark.table(SILVER_TBL)

print("Silver rows   :", silver_df.count())
print("Silver columns:", len(silver_df.columns))

display(silver_df.limit(10))

Silver rows   : 48000
Silver columns: 58


transaction_id,record_id,raw_id,brand,product_name,category,water_type,product_tier,bottle_size_ml,mrp,selling_price,discount_percent,cost_price,sales_units,marketing_spend,avg_rating,ratings_count,reviews_count,seller_rating,sales_channel,platform_source,seller_name,platform_url,stock_status,delivery_days,distributor_count,retailer_count,city,state,region,source_type,activity_date,ingestion_timestamp,bronze_ingestion_ts,extras_json,campaign_type,offer_type,promotion_flag,dq_error_reason,ingest_ts_eff,promotion_flag_std,offer_type_std,dq_crit_fail,is_valid,brand_std,discount_amount,discount_percent_calc,unit_profit,unit_margin_pct,gross_revenue,total_cost,gross_profit,event_date,event_year,event_month,event_day,silver_ingestion_ts,record_hash
001e8d9c-4160-4d9a-a574-34e8bad8fac2,4977,null,Aquafina,Aquafina 500ml,Packaged Drinking Water,RO,Mid,500,39.33,36.94,6.07,25.49,67,218.79,3.60,96,79,3.70,Online,Blinkit,Seller_5,https://example.com/product,IN_STOCK,4,2,101,Bengaluru,Karnataka,South,API,2024-11-30,2024-12-01T02:00:00.000Z,2026-03-13T05:41:18.860Z,"{""campaign_type"":""Always On"",""offer_type"":""Cashback"",""promotion_flag"":""False""}",Always On,Cashback,False,,2024-12-01T02:00:00.000Z,NO,NONE,0,true,AQUAFINA,2.39,6.08,11.45,31.00,2474.98,1707.83,767.15,2024-11-30,2024,11,30,2026-03-16T12:41:20.101Z,572b4c533ccbe02fdd9e39dedef6a82df071ce825e7384c87b5a4864bff3f30e
00291146-01dc-4ab5-b2e2-cb40f01a5598,26521,null,Rail Neer,Rail Neer 200ml,Packaged Drinking Water,Spring,Economy,200,12.49,9.37,24.97,8.25,149,194.27,4.80,241,155,4.40,Online,Bigbasket,Seller_5,https://example.com/product,IN_STOCK,2,3,40,Kolkata,West Bengal,East,API,2024-10-27,2024-10-27T01:00:00.000Z,2026-03-11T14:13:27.278Z,"{""campaign_type"":""Flash Sale"",""offer_type"":""BOGO"",""promotion_flag"":""False""}",Flash Sale,BOGO,False,,2024-10-27T01:00:00.000Z,NO,NONE,0,true,RAIL NEER,3.12,24.98,1.12,11.95,1396.13,1229.25,166.88,2024-10-27,2024,10,27,2026-03-16T12:41:20.101Z,fac2545e335f8ee10e6700bbda1dbc0f22c598f20481db4a5f6d2b4e47ca8434
0031fd3e-24bc-4d84-8573-4b859913acfa,4208,null,Vedica,Vedica 200ml,Packaged Drinking Water,Alkaline,Mid,200,20.34,18.98,6.70,11.30,69,155.49,3.70,144,66,4.60,Online,Amazon,Seller_7,https://example.com/product,IN_STOCK,5,3,38,Bengaluru,Karnataka,South,API,2024-10-30,2024-10-30T21:00:00.000Z,2026-03-11T14:13:39.951Z,"{""campaign_type"":""Festive"",""offer_type"":""Cashback"",""promotion_flag"":""True""}",Festive,Cashback,True,,2024-10-30T21:00:00.000Z,YES,CASHBACK,0,true,VEDICA,1.36,6.69,7.68,40.46,1309.62,779.70,529.92,2024-10-30,2024,10,30,2026-03-16T12:41:20.101Z,c09688cec9c047ee3a2db6a608068c5c05664ae3e460dcf97350dda6b2ed489a
0074e163-dd74-4179-b0e1-5d7a3cf7762f,48593,null,Aquafina,Aquafina 500ml,Packaged Drinking Water,Mineral,Premium,500,38.65,32.83,15.05,23.35,110,521.56,4.70,81,65,4.00,Online,Jiomart,Seller_17,https://example.com/product,IN_STOCK,2,4,34,Ahmedabad,Gujarat,West,API,2024-10-11,2024-10-12T11:00:00.000Z,2026-03-11T14:12:58.640Z,"{""campaign_type"":""Always On"",""offer_type"":""Discount"",""promotion_flag"":""True""}",Always On,Discount,True,,2024-10-12T11:00:00.000Z,YES,DISCOUNT,0,true,AQUAFINA,5.82,15.06,9.48,28.88,3611.30,2568.50,1042.80,2024-10-11,2024,10,11,2026-03-16T12:41:20.101Z,9631b067f4bd066cb869b4d6308d354a7f83100f22164816648227f4bb44b41f
008837a4-f939-4bb8-ac14-1976ff3d55f0,36052,null,Himalayan,Himalayan 200ml,Packaged Drinking Water,RO,Mid,200,23.78,20.74,12.80,15.40,151,395.24,4.30,266,213,4.10,Online,Blinkit,Seller_3,https://example.com/product,IN_STOCK,4,1,101,Indore,Madhya Pradesh,Central,API,2024-11-23,2024-11-24T16:00:00.000Z,2026-03-11T14:14:59.510Z,"{""campaign_type"":""Seasonal"",""offer_type"":""Discount"",""promotion_flag"":""False""}",Seasonal,Discount,False,,2024-11-24T16:00:00.000Z,NO,NONE,0,true,HIMALAYAN,3.04,12.78,5.34,25.75,3131.74,2325.40,806.34,2024-11-23,2024,11,23,2026-03-16T12:41:20.101Z,f785ec8d9e9f84af2d8c676d57ec6cdf33664c442f5f0a02c22940fb1db5ef7d
0091a668-160b-4139-a

Interpretation: Loads the Silver dataset that will be transformed into Gold.

## Gold Modeling Rules

This notebook uses the following design rules:

- Business keys are standardized before dimension creation
- Unknown members are added to every dimension
- Fact grain is one row per transaction_id
- Duplicate transaction versions are resolved using latest silver_ingestion_ts
- Gold KPI output tables are generated from the final fact table

In [0]:
required_cols = [
    "transaction_id",
    "record_hash",
    "silver_ingestion_ts",
    "event_date",
    "brand_std",
    "product_name",
    "category",
    "water_type",
    "product_tier",
    "bottle_size_ml",
    "region",
    "state",
    "city",
    "seller_name",
    "platform_source",
    "sales_channel",
    "source_type",
    "campaign_type",
    "offer_type_std",
    "promotion_flag_std",
    "mrp",
    "selling_price",
    "cost_price",
    "sales_units",
    "marketing_spend",
    "avg_rating",
    "ratings_count",
    "reviews_count",
    "delivery_days",
    "distributor_count",
    "retailer_count",
    "discount_percent",
    "gross_revenue",
    "total_cost",
    "gross_profit",
    "discount_amount"
]

missing_cols = [c for c in required_cols if c not in silver_df.columns]

if missing_cols:
    raise Exception(f"Gold load stopped. Missing required Silver columns: {missing_cols}")

print("Required column check passed.")

Required column check passed.


Interpretation:
 Stops the notebook early if Silver schema changes or required columns are missing.

## Gold Modeling Rules

This notebook follows the following rules:

- Business keys are standardized before dimension creation
- Unknown members are added to every dimension
- Fact grain is one row per transaction_id
- Duplicate transaction versions are resolved using latest silver_ingestion_ts
- Incremental loading is controlled through watermark logic
- Fact merge is protected by duplicate checks before merge
- KPI output tables are derived from the final fact table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DateType,
    StringType, BooleanType
)
from delta.tables import DeltaTable

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_DATE_TBL} (
    date_key      INT,
    event_date    DATE,
    day           INT,
    month         INT,
    month_name    STRING,
    quarter       INT,
    year          INT,
    week_of_year  INT,
    day_name      STRING,
    is_weekend    BOOLEAN
)
USING DELTA
""")

date_src = (
    silver_df
    .select(F.col("event_date").cast("date").alias("event_date"))
    .filter(F.col("event_date").isNotNull())
    .distinct()
    .withColumn("date_key", F.date_format("event_date", "yyyyMMdd").cast("int"))
    .withColumn("day", F.dayofmonth("event_date"))
    .withColumn("month", F.month("event_date"))
    .withColumn("month_name", F.date_format("event_date", "MMMM"))
    .withColumn("quarter", F.quarter("event_date"))
    .withColumn("year", F.year("event_date"))
    .withColumn("week_of_year", F.weekofyear("event_date"))
    .withColumn("day_name", F.date_format("event_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("event_date").isin([1, 7]))
    .select(
        "date_key", "event_date", "day", "month", "month_name",
        "quarter", "year", "week_of_year", "day_name", "is_weekend"
    )
)

unknown_date_schema = StructType([
    StructField("date_key", IntegerType(), True),
    StructField("event_date", DateType(), True),
    StructField("day", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("month_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("week_of_year", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("is_weekend", BooleanType(), True)
])

unknown_date = spark.createDataFrame(
    [(0, None, None, None, "UNKNOWN", None, None, None, "UNKNOWN", None)],
    schema=unknown_date_schema
)

date_src = date_src.unionByName(unknown_date)

(
    DeltaTable.forName(spark, DIM_DATE_TBL)
    .alias("t")
    .merge(date_src.alias("s"), "t.date_key = s.date_key")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("dim_date rows:", spark.table(DIM_DATE_TBL).count())
display(
    spark.table(DIM_DATE_TBL)
    .orderBy("date_key")
    .limit(10)
)

dim_date rows: 355


date_key,event_date,day,month,month_name,quarter,year,week_of_year,day_name,is_weekend
0,null,null,null,UNKNOWN,null,null,null,UNKNOWN,null
20240101,2024-01-01,1,1,January,1,2024,1,Monday,false
20240102,2024-01-02,2,1,January,1,2024,1,Tuesday,false
20240103,2024-01-03,3,1,January,1,2024,1,Wednesday,false
20240104,2024-01-04,4,1,January,1,2024,1,Thursday,false
20240105,2024-01-05,5,1,January,1,2024,1,Friday,false
20240106,2024-01-06,6,1,January,1,2024,1,Saturday,true
20240107,2024-01-07,7,1,January,1,2024,1,Sunday,true
20240108,2024-01-08,8,1,January,1,2024,2,Monday,false
20240109,2024-01-09,9,1,January,1,2024,2,Tuesday,false


Interpretation: 
Creates and loads the date dimension in the correct schema.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_PRODUCT_TBL} (
    product_key    STRING,
    brand          STRING,
    product_name   STRING,
    category       STRING,
    water_type     STRING,
    product_tier   STRING,
    pack_size_ml   INT
)
USING DELTA
""")

product_src = (
    silver_df
    .select(
        F.coalesce(F.col("brand_std"), F.lit("UNKNOWN")).cast("string").alias("brand"),
        F.coalesce(F.col("product_name"), F.lit("UNKNOWN")).cast("string").alias("product_name_raw"),
        F.coalesce(F.col("category"), F.lit("UNKNOWN")).cast("string").alias("category"),
        F.coalesce(F.col("water_type"), F.lit("UNKNOWN")).cast("string").alias("water_type"),
        F.coalesce(F.col("product_tier"), F.lit("UNKNOWN")).cast("string").alias("product_tier"),
        F.coalesce(F.col("bottle_size_ml"), F.lit(0)).cast("int").alias("pack_size_ml")
    )
    .withColumn("brand", norm_upper("brand"))
    .withColumn("product_name_raw", norm_upper("product_name_raw"))
    .withColumn("product_name", F.trim(F.regexp_replace(F.col("product_name_raw"), SIZE_TOKEN_RE, "")))
    .withColumn("product_name", F.trim(F.regexp_replace(F.col("product_name"), r"\s+", " ")))
    .withColumn("category", norm_upper("category"))
    .withColumn("water_type", norm_upper("water_type"))
    .withColumn("product_tier", norm_upper("product_tier"))
    .select("brand","product_name","category","water_type","product_tier","pack_size_ml")
    .distinct()
    .withColumn(
        "product_key",
        sha256_key(
            "brand",
            "product_name",
            "category",
            "water_type",
            "product_tier",
            F.col("pack_size_ml").cast("string")
        )
    )
    .select("product_key","brand","product_name","category","water_type","product_tier","pack_size_ml")
)

unknown_product = spark.createDataFrame(
    [("0", "UNKNOWN", "UNKNOWN", "UNKNOWN", "UNKNOWN", "UNKNOWN", 0)],
    ["product_key","brand","product_name","category","water_type","product_tier","pack_size_ml"]
)

product_src = product_src.unionByName(unknown_product)

DeltaTable.forName(spark, DIM_PRODUCT_TBL) \
    .alias("t") \
    .merge(product_src.alias("s"), "t.product_key = s.product_key") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("dim_product rows:", spark.table(DIM_PRODUCT_TBL).count())
display(spark.table(DIM_PRODUCT_TBL).orderBy("brand","product_name").limit(20))

dim_product rows: 418


product_key,brand,product_name,pack_size_ml
ac1445cea77770d6cd25948e9e61af0bd6c7c4b9149786d9aa6cccad556f97a3,AQUAFINA,AQUAFINA,1000
607001e2fdb56d37bea1ae2197321f9e1390b5b307058a062d325bdd50023cbf,AQUAFINA,AQUAFINA,200
649280083d2765596c1f3d22bbea51720ed237d047e67ea006ec5227fd2a022d,AQUAFINA,AQUAFINA,1000
5f31a333e553ea058dd54617627e84bba216fa17f6a80f2e88e2b8c294f1d1de,AQUAFINA,AQUAFINA,1000
0fb620c600a4908f767660ec709d60dfd073dda37090be68ac2afec05f10370d,AQUAFINA,AQUAFINA,2000
b634f4b1943b9c129b67e7ac465715c4132423639cde6fbc8ad24c8fec2713d0,AQUAFINA,AQUAFINA,2000
040e4314ec41c20c785b7e86a16dee1b41128df9d3f3700e3f9ce140b9ad1936,AQUAFINA,AQUAFINA,1000
caf4ec6260d8f14e500cb697f7270bf659b5b6d3063a15a67f0142a70b5a5a14,AQUAFINA,AQUAFINA,1000
aa02dfb1098a72795d75edc0b3f661516aaf7bf48e0bb4cd36139d6b51413f14,AQUAFINA,AQUAFINA,2000
2651cfd76366be0224459da942d59d0c2e7cd5b72730b4ed9b55aa6918a95a96,AQUAFINA,AQUAFINA,1000


Interpretation:
 Builds the product dimension using standardized product attributes and a deterministic SHA key.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_GEO_TBL} (
    geo_key   STRING,
    region    STRING,
    state     STRING,
    city      STRING
)
USING DELTA
""")

geo_src = (
    silver_df
    .select(
        F.coalesce(F.col("region"), F.lit("UNKNOWN")).cast("string").alias("region"),
        F.coalesce(F.col("state"), F.lit("UNKNOWN")).cast("string").alias("state"),
        F.coalesce(F.col("city"), F.lit("UNKNOWN")).cast("string").alias("city")
    )
    .withColumn("region", norm_upper("region"))
    .withColumn("state", norm_upper("state"))
    .withColumn("city", norm_upper("city"))
    .distinct()
    .withColumn("geo_key", sha256_key("region","state","city"))
    .select("geo_key","region","state","city")
)

unknown_geo = spark.createDataFrame(
    [("0", "UNKNOWN", "UNKNOWN", "UNKNOWN")],
    ["geo_key","region","state","city"]
)

geo_src = geo_src.unionByName(unknown_geo)

DeltaTable.forName(spark, DIM_GEO_TBL) \
    .alias("t") \
    .merge(geo_src.alias("s"), "t.geo_key = s.geo_key") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("dim_geo rows:", spark.table(DIM_GEO_TBL).count())
display(spark.table(DIM_GEO_TBL).orderBy("region","state","city").limit(20))

dim_geo rows: 14


geo_key,region,state,city
d5ced0aecdedaf86f0eff2f38f96e0d4b759cc0985303bec3d16d02948740797,CENTRAL,MADHYA PRADESH,BHOPAL
b376e1b4e48d34b43247c8c4cb86b909b93b0d83ff74e647ab0da4c56b406f4c,CENTRAL,MADHYA PRADESH,INDORE
62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,EAST,WEST BENGAL,KOLKATA
c48b62bfbc6cec3fda8005f1613e92a70119d7773245a60145c4ef9ba614ee24,NORTH,DELHI,DELHI
6795197aeabac1a08fe0559a5f4715fb06230833591904fc71f6535c72cd4b4f,NORTH,RAJASTHAN,JAIPUR
94d37358a05fc97dd44cb47fe96724535c88c37a737f641e750d48f44006d50a,SOUTH,KARNATAKA,BENGALURU
7fe20ae1fce2c9c22af1816b3d8b7fd3b2ebc2e63f7944f8669bc36276120dd3,SOUTH,TAMIL NADU,CHENNAI
350ff94fc7b8c8fb8bc3cf6f6819c7c369e7650417effe9b583599da5f0bead3,SOUTH,TELANGANA,HYDERABAD
0,UNKNOWN,UNKNOWN,UNKNOWN
UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN


Interpretation:
 Builds the geography dimension using normalized region, state, and city.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_CAMPAIGN_TBL} (
    campaign_key    STRING,
    campaign_name   STRING,
    campaign_type   STRING,
    offer_type      STRING,
    promotion_flag  STRING
)
USING DELTA
""")

campaign_src = (
    silver_df
    .select(
        F.coalesce(F.col("campaign_type"), F.lit("UNKNOWN")).cast("string").alias("campaign_type"),
        F.coalesce(F.col("offer_type_std"), F.lit("UNKNOWN")).cast("string").alias("offer_type"),
        F.coalesce(F.col("promotion_flag_std"), F.lit("UNKNOWN")).cast("string").alias("promotion_flag")
    )
    .withColumn("campaign_type", norm_upper("campaign_type"))
    .withColumn("offer_type", norm_upper("offer_type"))
    .withColumn("promotion_flag", norm_upper("promotion_flag"))
    .withColumn("campaign_name", F.col("campaign_type"))
    .distinct()
    .withColumn("campaign_key", sha256_key("campaign_name","campaign_type","offer_type","promotion_flag"))
    .select("campaign_key","campaign_name","campaign_type","offer_type","promotion_flag")
)

unknown_campaign = spark.createDataFrame(
    [("0", "UNKNOWN", "UNKNOWN", "UNKNOWN", "UNKNOWN")],
    ["campaign_key","campaign_name","campaign_type","offer_type","promotion_flag"]
)

campaign_src = campaign_src.unionByName(unknown_campaign)

DeltaTable.forName(spark, DIM_CAMPAIGN_TBL) \
    .alias("t") \
    .merge(campaign_src.alias("s"), "t.campaign_key = s.campaign_key") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("dim_campaign rows:", spark.table(DIM_CAMPAIGN_TBL).count())
display(spark.table(DIM_CAMPAIGN_TBL).orderBy("campaign_type","offer_type").limit(20))

dim_campaign rows: 26


campaign_key,campaign_name,campaign_type,offer_type,promotion_flag
6bdee57cca647245b6b63707dbbdd335e3b81529e7c94298ba47c4854cedfdd7,ALWAYS ON,ALWAYS ON,BOGO,YES
887132fa71b5bfd1918800f1c029b5752a6f4b50dcf74862a95d060185c6477f,ALWAYS ON,ALWAYS ON,CASHBACK,YES
694a723154adab62867c2d41699bd498721b7999484255aad115a2b807847837,ALWAYS ON,ALWAYS ON,DISCOUNT,YES
cc3358050f0f7f2137002540961729d260725d2ba82502aba39311f6073bc2de,ALWAYS ON,ALWAYS ON,NONE,NO
0b40ed73b449bc63611df9a63624040bb3032d723e490287e07af0074d1e593f,ALWAYS ON,ALWAYS ON,NONE,YES
eba8bd387d12ec189eb416b9613858c7dc4b03052e12f1311b0f9c328e6f97e3,ALWAYS ON,ALWAYS ON,UNKNOWN_OFFER,YES
5bf3429069624efaa86bf545b5b5b5c02386e902721138d48273a094f823327a,FESTIVE,FESTIVE,BOGO,YES
3d3c9cf383c4766afc21cd8ad5a8d3c9beef18577c2df883e2d3a35a7c46d5c2,FESTIVE,FESTIVE,CASHBACK,YES
18f767ce0b8c46197a636bc7a53cd44c84fe180a6e4e1cf014d89cd0ab4acb52,FESTIVE,FESTIVE,DISCOUNT,YES
29a240abdf511e2a5b3e11282f8299f77156c82e15fd63964e8a2be2a8ae0b06,FESTIVE,FESTIVE,NONE,YES


Interpretation:
 Builds the campaign dimension from campaign type, standardized offer type, and promotion flag.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_SELLER_CHANNEL_TBL} (
    seller_channel_key  STRING,
    seller              STRING,
    platform_source     STRING,
    channel             STRING,
    seller_type         STRING
)
USING DELTA
""")

seller_channel_src = (
    silver_df
    .select(
        F.coalesce(F.col("seller_name"), F.lit("UNKNOWN")).cast("string").alias("seller"),
        F.coalesce(F.col("platform_source"), F.lit("UNKNOWN")).cast("string").alias("platform_source"),
        F.coalesce(F.col("sales_channel"), F.lit("UNKNOWN")).cast("string").alias("channel"),
        F.coalesce(F.col("source_type"), F.lit("UNKNOWN")).cast("string").alias("seller_type")
    )
    .withColumn("seller", norm_upper("seller"))
    .withColumn("platform_source", norm_upper("platform_source"))
    .withColumn("channel", norm_upper("channel"))
    .withColumn("seller_type", norm_upper("seller_type"))
    .distinct()
    .withColumn("seller_channel_key", sha256_key("seller","platform_source","channel","seller_type"))
    .select("seller_channel_key","seller","platform_source","channel","seller_type")
)

unknown_seller_channel = spark.createDataFrame(
    [("0", "UNKNOWN", "UNKNOWN", "UNKNOWN", "UNKNOWN")],
    ["seller_channel_key","seller","platform_source","channel","seller_type"]
)

seller_channel_src = seller_channel_src.unionByName(unknown_seller_channel)

DeltaTable.forName(spark, DIM_SELLER_CHANNEL_TBL) \
    .alias("t") \
    .merge(seller_channel_src.alias("s"), "t.seller_channel_key = s.seller_channel_key") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("dim_seller_channel rows:", spark.table(DIM_SELLER_CHANNEL_TBL).count())
display(spark.table(DIM_SELLER_CHANNEL_TBL).orderBy("platform_source","seller").limit(20))

dim_seller_channel rows: 128


seller_channel_key,seller,platform_source,channel,seller_type
59877d42784d3569d365f50fef51d5b76982d5a77407f65cfc5e94e29ddd86cc,SELLER_1,AMAZON,ONLINE,API
d3a60045372283cc1249afdd685de5083fd04cb0bc55ca7a4f5e400a9c818324,SELLER_10,AMAZON,ONLINE,API
73d4aea8138f967f8c0bb915cb557f5ec1279ca2dd3ec8eb0155bcf64c266784,SELLER_11,AMAZON,ONLINE,API
e5865a27a084e5ecd167f24b93ba3b9be55e84daa7eaf12ecefe1f1d9c5dcd28,SELLER_12,AMAZON,ONLINE,API
c5196d09054feed4dd89415d4add0a24eba31e4a93890ab311710470bce7a70c,SELLER_13,AMAZON,ONLINE,API
3a5dcba07939aea4c64d83b822f8482ad0e61ff59efe1d8395a669fa4ce09709,SELLER_14,AMAZON,ONLINE,API
e5576442753b5cc8b0345337f45fc19e05c20df167155e30d803d899869dc88a,SELLER_15,AMAZON,ONLINE,API
5fd0ab76f9219cef8d864d96c57e58986257bf4a7876b24158093b9fa79691e7,SELLER_16,AMAZON,ONLINE,API
98a058fd6f0b7ed8518bd8160b8d410bd0b8365735ceba73dccf24ac45101f63,SELLER_17,AMAZON,ONLINE,API
b5c93ff9cd8f96e79da7c47b18d84ddff0fce79ed828305439dab0b041cfa262,SELLER_18,AMAZON,ONLINE,API


Interpretation:
 Creates the combined seller-channel dimension to match the PDF design exactly.

## Fact Table Build

The fact table stores:
- transaction grain
- foreign keys to all dimensions
- descriptive degenerate attributes
- measures
- KPI fields

Fact grain:
**one row per transaction_id**

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FACT_TBL} (
    fact_row_key            STRING,
    silver_ingestion_ts     TIMESTAMP,
    event_date              DATE,

    date_key                INT,
    product_key             STRING,
    geo_key                 STRING,
    campaign_key            STRING,
    seller_channel_key      STRING,

    sales_units             BIGINT,
    marketing_spend         DECIMAL(18,2),
    gross_revenue           DECIMAL(18,2),
    total_cost              DECIMAL(18,2),
    gross_profit            DECIMAL(18,2),
    discount_amount         DECIMAL(18,2),

    avg_rating              DECIMAL(10,2),
    ratings_count           BIGINT,
    reviews_count           BIGINT,
    stock_out_flag          BOOLEAN,
    delivery_days           DECIMAL(18,2),
    distributor_count       BIGINT,
    retailer_count          BIGINT,

    avg_mrp                 DECIMAL(18,2),
    avg_selling_price       DECIMAL(18,2),
    avg_cost_price          DECIMAL(18,2),
    avg_discount_percent    DECIMAL(18,4),

    campaign_revenue        DECIMAL(18,2),
    marketing_roi           DECIMAL(18,4),
    roas                    DECIMAL(18,4),
    cost_per_unit           DECIMAL(18,4),
    engagement_rate         DECIMAL(18,4),
    campaign_efficiency     DECIMAL(18,4)
)
USING DELTA
""")

wm = spark.table(FACT_TBL).agg(F.max("silver_ingestion_ts").alias("wm")).first()["wm"]

print("FULL_LOAD =", FULL_LOAD)
print("Current watermark =", wm)

FULL_LOAD = False
Current watermark = 2026-04-20 15:32:54.087373


Interpretation:
 Creates the fact table and reads the current watermark for incremental processing.

In [0]:
from pyspark.sql import functions as F

if FULL_LOAD or wm is None:
    src_new = silver_df
else:
    src_new = silver_df.filter(F.col("silver_ingestion_ts") >= F.lit(wm))

print("src_new rows:", src_new.count())
print("src_new distinct tx:", src_new.select("transaction_id").distinct().count())
print("Watermark used:", wm)

src_new rows: 37000
src_new distinct tx: 37000
Watermark used: 2026-04-20 15:32:54.087373


Interpretation: 
Selects either full source or incremental source depending on FULL_LOAD.

In [0]:
dim_date = spark.table(DIM_DATE_TBL).select("date_key", "event_date")

dim_product = (
    spark.table(DIM_PRODUCT_TBL)
    .select("product_key", "brand", "product_name", "pack_size_ml")
)

dim_geo = spark.table(DIM_GEO_TBL).select("geo_key", "region", "state", "city")

dim_campaign = spark.table(DIM_CAMPAIGN_TBL).select(
    "campaign_key", "campaign_name", "campaign_type", "offer_type", "promotion_flag"
)

dim_seller_channel = spark.table(DIM_SELLER_CHANNEL_TBL).select(
    "seller_channel_key", "seller", "platform_source", "channel", "seller_type"
)

Interpretation: 

This cell loads the already-created Gold dimension tables into temporary DataFrames so they can be used as lookup references during fact table construction.
These lookup DataFrames help map the standardized business attributes from the Silver source to their corresponding dimension keys in the fact table join logic.

In [0]:
base = (
    src_new
    .select(
        F.col("transaction_id").cast("string").alias("transaction_id"),
        F.col("record_hash").cast("string").alias("record_hash"),
        F.col("silver_ingestion_ts").cast("timestamp").alias("silver_ingestion_ts"),
        F.col("event_date").cast("date").alias("event_date"),

        F.coalesce(F.col("brand_std"), F.lit("UNKNOWN")).cast("string").alias("brand"),
        F.coalesce(F.col("product_name"), F.lit("UNKNOWN")).cast("string").alias("product_name_raw"),
        F.coalesce(F.col("bottle_size_ml"), F.lit(0)).cast("int").alias("pack_size_ml"),

        F.coalesce(F.col("region"), F.lit("UNKNOWN")).cast("string").alias("region"),
        F.coalesce(F.col("state"), F.lit("UNKNOWN")).cast("string").alias("state"),
        F.coalesce(F.col("city"), F.lit("UNKNOWN")).cast("string").alias("city"),

        F.coalesce(F.col("seller_name"), F.lit("UNKNOWN")).cast("string").alias("seller"),
        F.coalesce(F.col("platform_source"), F.lit("UNKNOWN")).cast("string").alias("platform_source"),
        F.coalesce(F.col("sales_channel"), F.lit("UNKNOWN")).cast("string").alias("channel"),
        F.coalesce(F.col("source_type"), F.lit("UNKNOWN")).cast("string").alias("seller_type"),

        F.coalesce(F.col("campaign_type"), F.lit("UNKNOWN")).cast("string").alias("campaign_type"),
        F.coalesce(F.col("offer_type_std"), F.lit("UNKNOWN")).cast("string").alias("offer_type"),
        F.coalesce(F.col("promotion_flag_std"), F.lit("UNKNOWN")).cast("string").alias("promotion_flag"),

        F.coalesce(F.col("mrp"), F.lit(0)).cast("decimal(18,2)").alias("mrp"),
        F.coalesce(F.col("selling_price"), F.lit(0)).cast("decimal(18,2)").alias("selling_price"),
        F.coalesce(F.col("cost_price"), F.lit(0)).cast("decimal(18,2)").alias("cost_price"),
        F.coalesce(F.col("sales_units"), F.lit(0)).cast("int").alias("sales_units"),
        F.coalesce(F.col("marketing_spend"), F.lit(0)).cast("decimal(18,2)").alias("marketing_spend"),
        F.col("avg_rating").cast("decimal(10,2)").alias("avg_rating"),
        F.col("ratings_count").cast("int").alias("ratings_count"),
        F.col("reviews_count").cast("int").alias("reviews_count"),

        F.when(F.col("stock_status") == "OUT_OF_STOCK", F.lit(True)).otherwise(F.lit(False)).alias("stock_out_flag"),
        F.col("delivery_days").cast("int").alias("delivery_days"),
        F.col("distributor_count").cast("int").alias("distributor_count"),
        F.col("retailer_count").cast("int").alias("retailer_count"),

        F.coalesce(F.col("discount_percent"), F.lit(0)).cast("decimal(18,4)").alias("discount_percent"),
        F.coalesce(F.col("gross_revenue"), F.lit(0)).cast("decimal(18,2)").alias("gross_revenue"),
        F.coalesce(F.col("total_cost"), F.lit(0)).cast("decimal(18,2)").alias("total_cost"),
        F.coalesce(F.col("gross_profit"), F.lit(0)).cast("decimal(18,2)").alias("gross_profit"),
        F.coalesce(F.col("discount_amount"), F.lit(0)).cast("decimal(18,2)").alias("discount_amount")
    )
    .withColumn("brand", norm_upper("brand"))
    .withColumn("product_name_raw", norm_upper("product_name_raw"))
    .withColumn("product_name", F.trim(F.regexp_replace(F.col("product_name_raw"), SIZE_TOKEN_RE, "")))
    .withColumn("product_name", F.trim(F.regexp_replace(F.col("product_name"), r"\s+", " ")))

    .withColumn("region", norm_upper("region"))
    .withColumn("state", norm_upper("state"))
    .withColumn("city", norm_upper("city"))

    .withColumn("seller", norm_upper("seller"))
    .withColumn("platform_source", norm_upper("platform_source"))
    .withColumn("channel", norm_upper("channel"))
    .withColumn("seller_type", norm_upper("seller_type"))

    .withColumn("campaign_type", norm_upper("campaign_type"))
    .withColumn("offer_type", norm_upper("offer_type"))
    .withColumn("promotion_flag", norm_upper("promotion_flag"))
    .withColumn("campaign_name", F.col("campaign_type"))

    .withColumn("product_key", sha256_key("brand", "product_name", F.col("pack_size_ml").cast("string")))
    .withColumn("geo_key", sha256_key("region", "state", "city"))
    .withColumn("campaign_key", sha256_key("campaign_name", "campaign_type", "offer_type", "promotion_flag"))
    .withColumn("seller_channel_key", sha256_key("seller", "platform_source", "channel", "seller_type"))
    .withColumn("date_key", F.when(F.col("event_date").isNull(), F.lit(0)).otherwise(F.date_format("event_date", "yyyyMMdd").cast("int")))
)

print("base rows:", base.count())
display(base.limit(20))

base rows: 37000


transaction_id,record_hash,silver_ingestion_ts,event_date,brand,product_name_raw,pack_size_ml,region,state,city,seller,platform_source,channel,seller_type,campaign_type,offer_type,promotion_flag,mrp,selling_price,cost_price,sales_units,marketing_spend,avg_rating,ratings_count,reviews_count,stock_out_flag,delivery_days,distributor_count,retailer_count,discount_percent,gross_revenue,total_cost,gross_profit,discount_amount,product_name,campaign_name,product_key,geo_key,campaign_key,seller_channel_key,date_key
000264cc-5544-431f-8852-6d5d88a79355,aacf72f0d7a4b225bc162ebcfd672c0a88cd2a5fcef50b0240cb3545a76703ed,2026-04-21T07:08:01.060Z,2024-03-17,BAILLEY,BAILLEY 200ML,200,NORTH,RAJASTHAN,JAIPUR,SELLER_16,FLIPKART,ONLINE,API,ALWAYS ON,UNKNOWN_OFFER,YES,21.67,21.25,16.01,198,574.08,4.70,51,49,false,4,4,61,1.9200,4207.50,3169.98,1037.52,0.42,BAILLEY,ALWAYS ON,077d6ea902b464bab8ced7fb169da98373c651f3246cea4f8234c545b14678cb,6795197aeabac1a08fe0559a5f4715fb06230833591904fc71f6535c72cd4b4f,eba8bd387d12ec189eb416b9613858c7dc4b03052e12f1311b0f9c328e6f97e3,241f33f45788109d06fa7316f607adf11dc39f0483372328e7ae95504d0bdde5,20240317
000a5e1f-2f52-4f4d-b6b0-e8446d44fdb5,990e1110b0d470bef0e46ba6d87b3f674ea0d36d799370a436047d8c40d451b8,2026-04-21T07:08:01.060Z,2024-04-30,KINLEY,KINLEY 2000ML,2000,WEST,GUJARAT,AHMEDABAD,SELLER_4,BIGBASKET,ONLINE,API,SEASONAL,NONE,NO,89.26,67.63,60.05,51,477.06,4.90,285,168,false,4,5,119,24.2300,3449.13,3062.55,386.58,21.63,KINLEY,SEASONAL,c19912a32b58bb85a7e1c10ec795498c3007139739c2992d977e3ac7907cf0ea,d3690e606cf2a90a3d4ace81b5a4922b4c520ff8365bff07a663eb80f94c4eaa,c97fcc670fdf89175daaa36aaf101be05e6c79642ce9086c5fb2f1ad93bd435e,e6edc723165d339211dbe6b9a551696556916ea19452ca1d5c63b195fc5f27eb,20240430
000a9067-b5f0-4b99-8bc8-df6a7d9fe6a7,cc2d39f39ed8751ee684371f95a06fb6ccca7dd6c80ce1cb96464b7bc56de4a2,2026-04-21T07:08:01.060Z,2024-05-15,HIMALAYAN,HIMALAYAN 200ML,200,CENTRAL,MADHYA PRADESH,INDORE,SELLER_4,SWIGGY INSTAMART,ONLINE,API,ALWAYS ON,BOGO,YES,15.89,14.29,9.93,100,171.46,4.00,284,238,true,1,1,37,10.0600,1429.00,993.00,436.00,1.60,HIMALAYAN,ALWAYS ON,2efdd6d6be98b75f0127c958521d3ee7fb004ab26c74ffb0856853e4a6ac71a2,b376e1b4e48d34b43247c8c4cb86b909b93b0d83ff74e647ab0da4c56b406f4c,6bdee57cca647245b6b63707dbbdd335e3b81529e7c94298ba47c4854cedfdd7,d243b45594e624259d46bfe4abff7daf520043104d92649fe392fb8bba51edec,20240515
000ccbf5-05ab-4b45-9e46-437ad181f3d6,b8df992604af1ff214951ed25739078afdebea6569107297b732d1c5945d11c9,2026-04-21T07:08:01.060Z,2024-04-25,HIMALAYAN,HIMALAYAN 2000ML,2000,CENTRAL,MADHYA PRADESH,INDORE,SELLER_1,JIOMART,ONLINE,API,FESTIVE,NONE,YES,75.85,73.26,44.18,151,1533.26,4.30,179,28,true,5,2,27,3.4100,11062.26,6671.18,4391.08,2.59,HIMALAYAN,FESTIVE,bd0936c79a72013ab2f9460ce3e197532a38d42284e8fc92a1df6b1aa1b86f5e,b376e1b4e48d34b43247c8c4cb86b909b93b0d83ff74e647ab0da4c56b406f4c,29a240abdf511e2a5b3e11282f8299f77156c82e15fd63964e8a2be2a8ae0b06,3bbffa715079ba5d6363ca62732b6d8f8a2aafc17d01a19a7dc1e30fc7c063d9,20240425
000d64b2-92cb-440c-8eb7-3659714ea73b,fd8600b0a195c3ff98da61691cbba9d918426b3e4f0a0cd8d7d0cdf15eaa604f,2026-04-21T07:08:01.060Z,2024-02-20,BAILLEY,BAILLEY 500ML,500,WEST,MAHARASHTRA,MUMBAI,SELLER_16,BIGBASKET,ONLINE,API,SEASONAL,NONE,NO,22.54,17.73,13.62,128,310.73,4.80,196,120,false,3,3,88,21.3600,2269.44,1743.36,526.08,4.81,BAILLEY,SEASONAL,2363648e08386ac9ba3abe6d3402b87e529f56f17a2b12384fce72a6da9bb03e,f5ed35492a84e9638f2164dd7f2e8142213b61e2b75b396ea8f171766cd211d0,c97fcc670fdf89175daaa36aaf101be05e6c79642ce9086c5fb2f1ad93bd435e,c1d4487fb7f4081a84fa9b8a8be1c03685a430cb3f2a5d3de3c4020827674a86,20240220
000eae96-a8f1-4654-9f8e-e7d31fdf8637,78ec2ca79249e84180a57e9f9dc14b468bdcad58001f0033646e58588376e541,2026-04-21T07:08:01.060Z,2024-09-09,VEDICA,VEDICA 2000ML,2000,WEST,MAHARASHTRA,PUNE,SELLER_15,BIGBASKET,ONLINE,API,SEASONAL,NONE,NO,76.29,75.60,48.51,164,2015.73,3.80,249,104,false,4,5,41,0.9100,12398.40,7955.64,4442.76,0.69,VEDICA,SEASONAL,f16804ad6334821fdcbf846ab5c3de6258c0ef4f942637

Interpretation: Standardizes source data, derives all business keys, and prepares the base fact source.

In [0]:
fact_src = (
    base
    .groupBy(
        "event_date",
        "date_key",
        "product_key",
        "geo_key",
        "campaign_key",
        "seller_channel_key"
    )
    .agg(
        F.max("silver_ingestion_ts").alias("silver_ingestion_ts"),
        F.sum("sales_units").cast("bigint").alias("sales_units"),
        F.sum("marketing_spend").cast("decimal(18,2)").alias("marketing_spend"),
        F.sum("gross_revenue").cast("decimal(18,2)").alias("gross_revenue"),
        F.sum("total_cost").cast("decimal(18,2)").alias("total_cost"),
        F.sum("gross_profit").cast("decimal(18,2)").alias("gross_profit"),
        F.sum("discount_amount").cast("decimal(18,2)").alias("discount_amount"),

        F.avg("avg_rating").cast("decimal(10,2)").alias("avg_rating"),
        F.sum("ratings_count").cast("bigint").alias("ratings_count"),
        F.sum("reviews_count").cast("bigint").alias("reviews_count"),

        F.max("stock_out_flag").alias("stock_out_flag"),
        F.avg("delivery_days").cast("decimal(18,2)").alias("delivery_days"),
        F.max("distributor_count").cast("bigint").alias("distributor_count"),
        F.max("retailer_count").cast("bigint").alias("retailer_count"),

        F.avg("mrp").cast("decimal(18,2)").alias("avg_mrp"),
        F.avg("selling_price").cast("decimal(18,2)").alias("avg_selling_price"),
        F.avg("cost_price").cast("decimal(18,2)").alias("avg_cost_price"),
        F.avg("discount_percent").cast("decimal(18,4)").alias("avg_discount_percent"),

        F.sum(
            F.when(F.col("promotion_flag") == "YES", F.col("gross_revenue")).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias("campaign_revenue")
    )
    .withColumn(
        "marketing_roi",
        F.round(
            F.when(F.col("marketing_spend") == 0, None)
             .otherwise((F.col("gross_profit") - F.col("marketing_spend")) / F.col("marketing_spend")),
            4
        )
    )
    .withColumn(
        "roas",
        F.round(
            F.when(F.col("marketing_spend") == 0, None)
             .otherwise(F.col("gross_revenue") / F.col("marketing_spend")),
            4
        )
    )
    .withColumn(
        "cost_per_unit",
        F.round(
            F.when(F.col("sales_units") == 0, None)
             .otherwise(F.col("total_cost") / F.col("sales_units")),
            4
        )
    )
    .withColumn(
        "engagement_rate",
        F.round(
            F.when((F.col("ratings_count") + F.col("reviews_count")) == 0, None)
             .otherwise(F.col("reviews_count") / (F.col("ratings_count") + F.col("reviews_count"))),
            4
        )
    )
    .withColumn("campaign_efficiency", F.col("roas"))
    .withColumn(
        "fact_row_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("date_key").cast("string"),
                F.col("product_key"),
                F.col("geo_key"),
                F.col("campaign_key"),
                F.col("seller_channel_key")
            ),
            256
        )
    )
    .select(
        "fact_row_key","silver_ingestion_ts","event_date",
        "date_key","product_key","geo_key","campaign_key","seller_channel_key",
        "sales_units","marketing_spend","gross_revenue","total_cost","gross_profit","discount_amount",
        "avg_rating","ratings_count","reviews_count","stock_out_flag","delivery_days",
        "distributor_count","retailer_count",
        "avg_mrp","avg_selling_price","avg_cost_price","avg_discount_percent",
        "campaign_revenue","marketing_roi","roas","cost_per_unit","engagement_rate","campaign_efficiency"
    )
)

print("fact_src rows:", fact_src.count())
display(fact_src.limit(20))

fact_src rows: 36992


fact_row_key,silver_ingestion_ts,event_date,date_key,product_key,geo_key,campaign_key,seller_channel_key,sales_units,marketing_spend,gross_revenue,total_cost,gross_profit,discount_amount,avg_rating,ratings_count,reviews_count,stock_out_flag,delivery_days,distributor_count,retailer_count,avg_mrp,avg_selling_price,avg_cost_price,avg_discount_percent,campaign_revenue,marketing_roi,roas,cost_per_unit,engagement_rate,campaign_efficiency
101040940bb2c712ed46dabbb44fc067a0a7cb5eaa9fc89f14ebcd843f020eb5,2026-04-21T07:08:01.060Z,2024-02-05,20240205,4925ee45fb004629a81aeea660fc8a9ba3dd398698fad31fd359ffc6986c6ea3,94d37358a05fc97dd44cb47fe96724535c88c37a737f641e750d48f44006d50a,48145babe4c82539bc4ed6ab3251f484bf7d9c6e4b6567995ca13b03ac1152bc,204758f6cd5a2bddb46867daa9c7f00dbb4394df432550d474d0a6d5f27d4192,181,2154.04,12590.36,10213.83,2376.53,7.56,4.90,293,165,false,3.00,3,43,77.12,69.56,56.43,9.8000,0.00,0.1033,5.8450,56.4300,0.3603,5.8450
b4ef38b893c672b9eceeb1431429f23e6c57c77653f4c3652497797d12cab1ee,2026-04-21T07:08:01.060Z,2024-07-19,20240719,791df55ba45fc4d19dc2d71ca4e604ab31e60df3acc5b9674fad78751270b1e6,510e124afff6d725fd04c3e3346a679a918f731738862eb92a5b8fd4329e0bab,0f5fe86b92f33da71fdac0b1d949829577d6ea3f591383ed0a433d4e5c116f5b,0ccdcfbc703aadf1bbc1f57f75ac14332114722f2524c50e7e72e28ada5d5372,151,454.08,4839.55,4708.18,131.37,9.66,4.30,194,23,false,4.00,5,0,41.71,32.05,31.18,23.1500,4839.55,-0.7107,10.6579,31.1800,0.106,10.6579
cb0fbb92c4fae94bdbf320b425f7f8a5535eada9d2f80ed577a2561260c5b422,2026-04-21T07:08:01.060Z,2024-01-05,20240105,898f173ccb7274f2434c703674e28e1cc90c357bd841f76eb75d10f53db16591,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,3d3c9cf383c4766afc21cd8ad5a8d3c9beef18577c2df883e2d3a35a7c46d5c2,74a1936437cff16ec01ad4c3b940a531cb201f6ce60663a2ffec03cfe1d1f700,31,89.65,850.02,573.50,276.52,0.85,4.80,189,188,false,1.00,5,65,28.27,27.42,18.50,3.0100,850.02,2.0844,9.4815,18.5000,0.4987,9.4815
5eabee10087fda606dc21f9a511ebabd2dba6fde94a951a0e498193d783f665a,2026-04-21T07:08:01.060Z,2024-03-01,20240301,a88f591ccdcb3e6912ad622574bd1e03c6c7780832a096a152396347dcaf18c2,b376e1b4e48d34b43247c8c4cb86b909b93b0d83ff74e647ab0da4c56b406f4c,887132fa71b5bfd1918800f1c029b5752a6f4b50dcf74862a95d060185c6477f,f91c1cf6f04a5f1d0b9ca7c9f5d21eba177bd6d1650148ce2d23b667fc15ba79,74,280.46,1684.24,1229.14,455.10,1.56,3.60,116,24,false,5.00,1,16,24.32,22.76,16.61,6.4100,1684.24,0.6227,6.0053,16.6100,0.1714,6.0053
9dede52c261b20a978342159dc6e336ff42404c9cb126660bfefa23de074a65e,2026-04-21T07:08:01.060Z,2024-05-31,20240531,218f307948926de3ac97fe097658f5a1c5cad5ed823b3fe9fd9939f013eafed4,b376e1b4e48d34b43247c8c4cb86b909b93b0d83ff74e647ab0da4c56b406f4c,5bf3429069624efaa86bf545b5b5b5c02386e902721138d48273a094f823327a,a20b5310d1b5b4d26e79d9e18a8b856ef53ea25f90fe1d9050ef2aa7497eca07,183,553.03,3974.76,3398.31,576.45,3.76,3.80,58,26,false,4.00,2,28,25.48,21.72,18.57,14.7500,3974.76,0.0423,7.1872,18.5700,0.3095,7.1872
cad66b05378d2a73a6dd705c6f113f3283c07a9d55d552b5079e64957c96cfb6,2026-04-21T07:08:01.060Z,2024-03-21,20240321,077d6ea902b464bab8ced7fb169da98373c651f3246cea4f8234c545b14678cb,5a0b1e99e13b238c9c1067f64ea8c8af5d52991d6c308f8a5d56a8cab7ea3f24,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,bd1333254858910279aa39620b52806fa718dbcc13c8aabaedbee039719c4ceb,114,273.05,2000.70,1397.64,603.06,1.64,4.00,188,141,false,3.00,5,62,19.19,17.55,12.26,8.5700,0.00,1.2086,7.3272,12.2600,0.4286,7.3272
961d31a3a211e4dc115fc8162282d8675cc9a84ab0db217f62805f3716bd8e1c,2026-04-21T07:08:01.060Z,2024-06-26,20240626,a88f591ccdcb3e6912ad622574bd1e03c6c7780832a096a152396347dcaf18c2,7fe20ae1fce2c9c22af1816b3d8b7fd3b2ebc2e63f7944f8669bc36276120dd3,48145babe4c82539bc4ed6ab3251f484bf7d9c6e4b6567995ca13b03ac1152bc,d42de17b952bdf2f0cc587915ecc6a9043531fc92dd1d1a4e3ca4e87904ec4bd,65,109.96,1353.30,1131.65,221.65,3.75,5.00,161,109,false,5.00,1,39,24.57,20.82,17.41,15.2800,0.00,1.0157,12.3072,17.4100,0.4037,12.3072
4f1ce2f199f0126688528b9fe

Interpretation: Computes KPI fields and enforces fact grain using latest silver_ingestion_ts.

In [0]:
from delta.tables import DeltaTable

dup_fact = (
    fact_src.groupBy("fact_row_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if dup_fact > 0:
    raise Exception(f"Gold load stopped. Duplicate fact_row_key found before merge: {dup_fact}")

print("Pre-merge fact_row_key uniqueness check passed.")

tgt = DeltaTable.forName(spark, FACT_TBL)

merge_set = {
    "silver_ingestion_ts": "s.silver_ingestion_ts",
    "event_date": "s.event_date",
    "date_key": "s.date_key",
    "product_key": "s.product_key",
    "geo_key": "s.geo_key",
    "campaign_key": "s.campaign_key",
    "seller_channel_key": "s.seller_channel_key",
    "sales_units": "s.sales_units",
    "marketing_spend": "s.marketing_spend",
    "gross_revenue": "s.gross_revenue",
    "total_cost": "s.total_cost",
    "gross_profit": "s.gross_profit",
    "discount_amount": "s.discount_amount",
    "avg_rating": "s.avg_rating",
    "ratings_count": "s.ratings_count",
    "reviews_count": "s.reviews_count",
    "stock_out_flag": "s.stock_out_flag",
    "delivery_days": "s.delivery_days",
    "distributor_count": "s.distributor_count",
    "retailer_count": "s.retailer_count",
    "avg_mrp": "s.avg_mrp",
    "avg_selling_price": "s.avg_selling_price",
    "avg_cost_price": "s.avg_cost_price",
    "avg_discount_percent": "s.avg_discount_percent",
    "campaign_revenue": "s.campaign_revenue",
    "marketing_roi": "s.marketing_roi",
    "roas": "s.roas",
    "cost_per_unit": "s.cost_per_unit",
    "engagement_rate": "s.engagement_rate",
    "campaign_efficiency": "s.campaign_efficiency"
}

(
    tgt.alias("t")
    .merge(fact_src.alias("s"), "t.fact_row_key = s.fact_row_key")
    .whenMatchedUpdate(set=merge_set)
    .whenNotMatchedInsert(values={
        "fact_row_key": "s.fact_row_key",
        **merge_set
    })
    .execute()
)

print("fact_marketing_daily rows:", spark.table(FACT_TBL).count())
display(spark.table(FACT_TBL).limit(20))

Pre-merge fact_row_key uniqueness check passed.
fact_marketing_daily rows: 47991


fact_row_key,silver_ingestion_ts,event_date,date_key,product_key,geo_key,campaign_key,seller_channel_key,sales_units,marketing_spend,gross_revenue,total_cost,gross_profit,discount_amount,avg_rating,ratings_count,reviews_count,stock_out_flag,delivery_days,distributor_count,retailer_count,avg_mrp,avg_selling_price,avg_cost_price,avg_discount_percent,campaign_revenue,marketing_roi,roas,cost_per_unit,engagement_rate,campaign_efficiency
b3074fa25a34f365acebe0f1d109491bcb67797d1b965f91f269fc409d1446ac,2026-03-16T12:41:20.101Z,2024-11-19,20241119,f16804ad6334821fdcbf846ab5c3de6258c0ef4f942637197896f7e53cc9805a,d3690e606cf2a90a3d4ace81b5a4922b4c520ff8365bff07a663eb80f94c4eaa,6bdee57cca647245b6b63707dbbdd335e3b81529e7c94298ba47c4854cedfdd7,dbc0d4fa0ea9759824f63fc5fc34d5ac8eefcad17720d057687dcd20e51c25a0,191,2232.48,17587.28,12787.45,4799.83,8.39,4.00,53,20,true,5.00,1,48,100.47,92.08,66.95,8.3500,17587.28,1.1500,7.8779,66.9500,0.2740,7.8779
9bbaebc6ddfe0af11abae57ad6a327048b990d69b42e4c0947e0fef1db53fb72,2026-03-16T12:41:20.101Z,2024-10-06,20241006,898f173ccb7274f2434c703674e28e1cc90c357bd841f76eb75d10f53db16591,f5ed35492a84e9638f2164dd7f2e8142213b61e2b75b396ea8f171766cd211d0,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,7a03e650e89777d88242de56e1dd6d708b4e09a69dbed54b6fa7862c17bbae2b,147,498.87,5819.73,4257.12,1562.61,4.61,3.70,223,189,false,2.00,2,63,44.20,39.59,28.96,10.4200,0.00,2.1323,11.6658,28.9600,0.4587,11.6658
f9de463cce70730d963fc8fe818109fde02f20440870819c7c3b74bcbe4feefd,2026-03-16T12:41:20.101Z,2024-10-13,20241013,2c6fd7b138cc159510558f84d5622446b6df5439883f58c01db74a7572888ad2,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,13955b64a6bfa57bef2efc5d2224553cdfe435d157944fb4a62f20b82ed69662,100,191.37,2911.00,2224.00,687.00,1.63,3.80,53,50,false,1.00,3,97,30.74,29.11,22.24,5.2900,0.00,2.5899,15.2114,22.2400,0.4854,15.2114
9cd3f6554dd91297d601e0317fb58ec873e35224323905765d11e8db618d2d37,2026-03-16T12:41:20.101Z,2024-11-23,20241123,bd0936c79a72013ab2f9460ce3e197532a38d42284e8fc92a1df6b1aa1b86f5e,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,1847218e97680115e7c7d774c1e3b2bdc41bcad73940987e473b162354d83d55,128,955.09,9891.84,7339.52,2552.32,3.90,4.60,300,119,false,3.00,3,15,81.18,77.28,57.34,4.8000,0.00,1.6723,10.3570,57.3400,0.2840,10.3570
b3d11a6f6b00dbedf3f338288a6e2250c0576dc9c2bd86acad4e96f4843d0b35,2026-03-16T12:41:20.101Z,2024-10-30,20241030,ffdbd6dbbcce5df8c4767ade9be2b9ad24185cbe1134b69db6925dcce1e10031,6795197aeabac1a08fe0559a5f4715fb06230833591904fc71f6535c72cd4b4f,694a723154adab62867c2d41699bd498721b7999484255aad115a2b807847837,eea5f8632bf6ddc0a54ca39ebbf89f2621d0ff80ef58d3575758a2d31e3c89a2,190,607.13,3902.60,2492.80,1409.80,0.16,5.00,230,229,false,2.00,3,87,20.70,20.54,13.12,0.7900,3902.60,1.3221,6.4279,13.1200,0.4989,6.4279
93350718185a041756609b60b47ecede525b58707e3830236277ace8a66f6863,2026-03-16T12:41:20.101Z,2024-10-23,20241023,5b7fbf0d977c7ba06c641b15a98e965740141dffa31dcf5c313937ab09bdac41,d5ced0aecdedaf86f0eff2f38f96e0d4b759cc0985303bec3d16d02948740797,694a723154adab62867c2d41699bd498721b7999484255aad115a2b807847837,d3a60045372283cc1249afdd685de5083fd04cb0bc55ca7a4f5e400a9c818324,180,374.03,5941.80,3925.80,2016.00,5.47,4.90,275,103,false,5.00,3,17,38.48,33.01,21.81,14.2100,5941.80,4.3899,15.8859,21.8100,0.2725,15.8859
f394d667fa64ed0100df3f93a9edd76c8ceb79d45315c34abb6af58f0605fcac,2026-03-16T12:41:20.101Z,2024-11-14,20241114,bd0936c79a72013ab2f9460ce3e197532a38d42284e8fc92a1df6b1aa1b86f5e,7fe20ae1fce2c9c22af1816b3d8b7fd3b2ebc2e63f7944f8669bc36276120dd3,5cb03ed508e89f9f918a9c0f7ccbbb99cf4017040484c16b6ac123e4a865a99e,b99ce08ebc2b20833c2122fec734ff5517127460af08df94917ecdae821b1ec4,102,991.10,6114.90,4371.72,1743.18,2.17,4.70,99,38,false,3.00,4,58,62.12,59.95,42.86,3.4900,6114.90,0.7588,6.1698,42.8600,0.2774,6.1698
1d5dc75235

Interpretation: Prevents merge failures by ensuring source uniqueness on both transaction and record hash.

In [0]:
fact_df = spark.table(FACT_TBL)

# Read dimensions safely
dim_product_df = spark.table(DIM_PRODUCT_TBL)
dim_geo_df = spark.table(DIM_GEO_TBL)
dim_campaign_df = spark.table(DIM_CAMPAIGN_TBL)
dim_seller_channel_df = spark.table(DIM_SELLER_CHANNEL_TBL)

# Ensure missing descriptive columns exist in dim_product for KPI layer
if "product_tier" not in dim_product_df.columns:
    dim_product_df = dim_product_df.withColumn("product_tier", F.lit("UNKNOWN"))
if "water_type" not in dim_product_df.columns:
    dim_product_df = dim_product_df.withColumn("water_type", F.lit("UNKNOWN"))
if "category" not in dim_product_df.columns:
    dim_product_df = dim_product_df.withColumn("category", F.lit("UNKNOWN"))

dim_product_df = dim_product_df.select(
    "product_key",
    F.coalesce(F.col("brand"), F.lit("UNKNOWN")).alias("brand"),
    F.coalesce(F.col("product_tier"), F.lit("UNKNOWN")).alias("product_tier"),
    F.coalesce(F.col("water_type"), F.lit("UNKNOWN")).alias("water_type")
)

dim_geo_df = dim_geo_df.select(
    "geo_key",
    F.coalesce(F.col("region"), F.lit("UNKNOWN")).alias("region"),
    F.coalesce(F.col("state"), F.lit("UNKNOWN")).alias("state"),
    F.coalesce(F.col("city"), F.lit("UNKNOWN")).alias("city")
)

dim_campaign_df = dim_campaign_df.select(
    "campaign_key",
    F.coalesce(F.col("campaign_type"), F.lit("UNKNOWN")).alias("campaign_type"),
    F.coalesce(F.col("offer_type"), F.lit("UNKNOWN")).alias("offer_type"),
    F.coalesce(F.col("promotion_flag"), F.lit("UNKNOWN")).alias("promotion_flag")
)

dim_seller_channel_df = dim_seller_channel_df.select(
    "seller_channel_key",
    F.coalesce(F.col("platform_source"), F.lit("UNKNOWN")).alias("platform_source"),
    F.coalesce(F.col("channel"), F.lit("UNKNOWN")).alias("channel")
)

fact_enriched = (
    fact_df
    .join(dim_product_df, "product_key", "left")
    .join(dim_geo_df, "geo_key", "left")
    .join(dim_campaign_df, "campaign_key", "left")
    .join(dim_seller_channel_df, "seller_channel_key", "left")
    .withColumn("brand", F.coalesce(F.col("brand"), F.lit("UNKNOWN")))
    .withColumn("product_tier", F.coalesce(F.col("product_tier"), F.lit("UNKNOWN")))
    .withColumn("water_type", F.coalesce(F.col("water_type"), F.lit("UNKNOWN")))
    .withColumn("region", F.coalesce(F.col("region"), F.lit("UNKNOWN")))
    .withColumn("state", F.coalesce(F.col("state"), F.lit("UNKNOWN")))
    .withColumn("city", F.coalesce(F.col("city"), F.lit("UNKNOWN")))
    .withColumn("campaign_type", F.coalesce(F.col("campaign_type"), F.lit("UNKNOWN")))
    .withColumn("offer_type", F.coalesce(F.col("offer_type"), F.lit("UNKNOWN")))
    .withColumn("promotion_flag", F.coalesce(F.col("promotion_flag"), F.lit("UNKNOWN")))
    .withColumn("platform_source", F.coalesce(F.col("platform_source"), F.lit("UNKNOWN")))
    .withColumn("channel", F.coalesce(F.col("channel"), F.lit("UNKNOWN")))
)

brand_kpi = (
    fact_enriched
    .groupBy("brand")
    .agg(
        F.sum("sales_units").alias("total_sales_units"),
        F.sum("gross_revenue").alias("total_revenue"),
        F.sum("gross_profit").alias("total_profit"),
        F.avg("marketing_roi").alias("avg_marketing_roi"),
        F.avg("roas").alias("avg_roas")
    )
)
brand_kpi.write.format("delta").mode("overwrite").saveAsTable(KPI_BRAND_TBL)

campaign_kpi = (
    fact_enriched
    .groupBy("campaign_type", "offer_type", "promotion_flag")
    .agg(
        F.sum("sales_units").alias("total_sales_units"),
        F.sum("campaign_revenue").alias("total_campaign_revenue"),
        F.avg("campaign_efficiency").alias("avg_campaign_efficiency"),
        F.avg("marketing_roi").alias("avg_marketing_roi"),
        F.avg("roas").alias("avg_roas")
    )
)
campaign_kpi.write.format("delta").mode("overwrite").saveAsTable(KPI_CAMPAIGN_TBL)

platform_kpi = (
    fact_enriched
    .groupBy("platform_source", "channel")
    .agg(
        F.sum("sales_units").alias("total_sales_units"),
        F.sum("gross_revenue").alias("total_revenue"),
        F.avg("avg_rating").alias("avg_rating"),
        F.avg("roas").alias("avg_roas")
    )
)
platform_kpi.write.format("delta").mode("overwrite").saveAsTable(KPI_PLATFORM_TBL)

geo_kpi = (
    fact_enriched
    .groupBy("region", "state", "city")
    .agg(
        F.sum("sales_units").alias("total_sales_units"),
        F.sum("gross_revenue").alias("total_revenue"),
        F.sum("gross_profit").alias("total_profit"),
        F.avg("marketing_roi").alias("avg_marketing_roi"),
        F.avg("roas").alias("avg_roas")
    )
)
geo_kpi.write.format("delta").mode("overwrite").saveAsTable(KPI_GEO_TBL)

tier_kpi = (
    fact_enriched
    .groupBy("product_tier", "brand")
    .agg(
        F.sum("sales_units").alias("total_sales_units"),
        F.sum("gross_revenue").alias("total_revenue"),
        F.sum("gross_profit").alias("total_profit"),
        F.avg("marketing_roi").alias("avg_marketing_roi"),
        F.avg("roas").alias("avg_roas")
    )
)
tier_kpi.write.format("delta").mode("overwrite").saveAsTable(KPI_TIER_TBL)

water_kpi = (
    fact_enriched
    .groupBy("water_type")
    .agg(
        F.sum("sales_units").alias("total_sales_units"),
        F.sum("gross_revenue").alias("total_revenue"),
        F.sum("gross_profit").alias("total_profit"),
        F.avg("marketing_roi").alias("avg_marketing_roi"),
        F.avg("roas").alias("avg_roas")
    )
)
water_kpi.write.format("delta").mode("overwrite").saveAsTable(KPI_WATER_TBL)

monthly_kpi = (
    fact_enriched
    .groupBy(F.year("event_date").alias("year"), F.month("event_date").alias("month"))
    .agg(
        F.sum("sales_units").alias("total_sales_units"),
        F.sum("gross_revenue").alias("total_revenue"),
        F.sum("gross_profit").alias("total_profit"),
        F.avg("marketing_roi").alias("avg_marketing_roi"),
        F.avg("roas").alias("avg_roas")
    )
)
monthly_kpi.write.format("delta").mode("overwrite").saveAsTable(KPI_MONTHLY_TBL)

print("KPI output tables created successfully.")

KPI output tables created successfully.


Interpretation: Builds the KPI output tables 

In [0]:
fact_df = spark.table(FACT_TBL)

total_rows = fact_df.count()
distinct_fact = fact_df.select("fact_row_key").distinct().count()
dup_fact_rows = total_rows - distinct_fact

print("fact rows                 :", total_rows)
print("distinct fact_row_key     :", distinct_fact)
print("duplicate fact rows       :", dup_fact_rows)

print("null date_key             :", fact_df.filter(F.col("date_key").isNull()).count())
print("null product_key          :", fact_df.filter(F.col("product_key").isNull()).count())
print("null geo_key              :", fact_df.filter(F.col("geo_key").isNull()).count())
print("null campaign_key         :", fact_df.filter(F.col("campaign_key").isNull()).count())
print("null seller_channel_key   :", fact_df.filter(F.col("seller_channel_key").isNull()).count())

display(fact_df.limit(20))

fact rows                 : 47991
distinct fact_row_key     : 47991
duplicate fact rows       : 0
null date_key             : 0
null product_key          : 0
null geo_key              : 0
null campaign_key         : 0
null seller_channel_key   : 0


fact_row_key,silver_ingestion_ts,event_date,date_key,product_key,geo_key,campaign_key,seller_channel_key,sales_units,marketing_spend,gross_revenue,total_cost,gross_profit,discount_amount,avg_rating,ratings_count,reviews_count,stock_out_flag,delivery_days,distributor_count,retailer_count,avg_mrp,avg_selling_price,avg_cost_price,avg_discount_percent,campaign_revenue,marketing_roi,roas,cost_per_unit,engagement_rate,campaign_efficiency
b3074fa25a34f365acebe0f1d109491bcb67797d1b965f91f269fc409d1446ac,2026-03-16T12:41:20.101Z,2024-11-19,20241119,f16804ad6334821fdcbf846ab5c3de6258c0ef4f942637197896f7e53cc9805a,d3690e606cf2a90a3d4ace81b5a4922b4c520ff8365bff07a663eb80f94c4eaa,6bdee57cca647245b6b63707dbbdd335e3b81529e7c94298ba47c4854cedfdd7,dbc0d4fa0ea9759824f63fc5fc34d5ac8eefcad17720d057687dcd20e51c25a0,191,2232.48,17587.28,12787.45,4799.83,8.39,4.00,53,20,true,5.00,1,48,100.47,92.08,66.95,8.3500,17587.28,1.1500,7.8779,66.9500,0.2740,7.8779
9bbaebc6ddfe0af11abae57ad6a327048b990d69b42e4c0947e0fef1db53fb72,2026-03-16T12:41:20.101Z,2024-10-06,20241006,898f173ccb7274f2434c703674e28e1cc90c357bd841f76eb75d10f53db16591,f5ed35492a84e9638f2164dd7f2e8142213b61e2b75b396ea8f171766cd211d0,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,7a03e650e89777d88242de56e1dd6d708b4e09a69dbed54b6fa7862c17bbae2b,147,498.87,5819.73,4257.12,1562.61,4.61,3.70,223,189,false,2.00,2,63,44.20,39.59,28.96,10.4200,0.00,2.1323,11.6658,28.9600,0.4587,11.6658
f9de463cce70730d963fc8fe818109fde02f20440870819c7c3b74bcbe4feefd,2026-03-16T12:41:20.101Z,2024-10-13,20241013,2c6fd7b138cc159510558f84d5622446b6df5439883f58c01db74a7572888ad2,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,13955b64a6bfa57bef2efc5d2224553cdfe435d157944fb4a62f20b82ed69662,100,191.37,2911.00,2224.00,687.00,1.63,3.80,53,50,false,1.00,3,97,30.74,29.11,22.24,5.2900,0.00,2.5899,15.2114,22.2400,0.4854,15.2114
9cd3f6554dd91297d601e0317fb58ec873e35224323905765d11e8db618d2d37,2026-03-16T12:41:20.101Z,2024-11-23,20241123,bd0936c79a72013ab2f9460ce3e197532a38d42284e8fc92a1df6b1aa1b86f5e,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,1847218e97680115e7c7d774c1e3b2bdc41bcad73940987e473b162354d83d55,128,955.09,9891.84,7339.52,2552.32,3.90,4.60,300,119,false,3.00,3,15,81.18,77.28,57.34,4.8000,0.00,1.6723,10.3570,57.3400,0.2840,10.3570
b3d11a6f6b00dbedf3f338288a6e2250c0576dc9c2bd86acad4e96f4843d0b35,2026-03-16T12:41:20.101Z,2024-10-30,20241030,ffdbd6dbbcce5df8c4767ade9be2b9ad24185cbe1134b69db6925dcce1e10031,6795197aeabac1a08fe0559a5f4715fb06230833591904fc71f6535c72cd4b4f,694a723154adab62867c2d41699bd498721b7999484255aad115a2b807847837,eea5f8632bf6ddc0a54ca39ebbf89f2621d0ff80ef58d3575758a2d31e3c89a2,190,607.13,3902.60,2492.80,1409.80,0.16,5.00,230,229,false,2.00,3,87,20.70,20.54,13.12,0.7900,3902.60,1.3221,6.4279,13.1200,0.4989,6.4279
93350718185a041756609b60b47ecede525b58707e3830236277ace8a66f6863,2026-03-16T12:41:20.101Z,2024-10-23,20241023,5b7fbf0d977c7ba06c641b15a98e965740141dffa31dcf5c313937ab09bdac41,d5ced0aecdedaf86f0eff2f38f96e0d4b759cc0985303bec3d16d02948740797,694a723154adab62867c2d41699bd498721b7999484255aad115a2b807847837,d3a60045372283cc1249afdd685de5083fd04cb0bc55ca7a4f5e400a9c818324,180,374.03,5941.80,3925.80,2016.00,5.47,4.90,275,103,false,5.00,3,17,38.48,33.01,21.81,14.2100,5941.80,4.3899,15.8859,21.8100,0.2725,15.8859
f394d667fa64ed0100df3f93a9edd76c8ceb79d45315c34abb6af58f0605fcac,2026-03-16T12:41:20.101Z,2024-11-14,20241114,bd0936c79a72013ab2f9460ce3e197532a38d42284e8fc92a1df6b1aa1b86f5e,7fe20ae1fce2c9c22af1816b3d8b7fd3b2ebc2e63f7944f8669bc36276120dd3,5cb03ed508e89f9f918a9c0f7ccbbb99cf4017040484c16b6ac123e4a865a99e,b99ce08ebc2b20833c2122fec734ff5517127460af08df94917ecdae821b1ec4,102,991.10,6114.90,4371.72,1743.18,2.17,4.70,99,38,false,3.00,4,58,62.12,59.95,42.86,3.4900,6114.90,0.7588,6.1698,42.8600,0.2774,6.1698
1d5dc75235

Interpretation: Final validation of fact grain, uniqueness and sample KPI review.

## Gold Layer Conclusion

The Gold layer has been modeled in a PDF-aligned format with:

- 5 dimension tables
- 1 central fact table
- KPI output tables for analytical reporting

This model is now ready for downstream migration into:
**SSMS → Mart Layer → Power BI**

# Below is some extra cells for table verification


In [0]:
# ============================================
# GOLD OUTPUT VISUALIZATION CELL
# Displays all dimension, fact, and KPI tables
# ============================================

CATALOG = "workspace"
SCHEMA = "water_bottle_db"

DIM_DATE_TBL           = f"{CATALOG}.{SCHEMA}.dim_date"
DIM_PRODUCT_TBL        = f"{CATALOG}.{SCHEMA}.dim_product"
DIM_GEO_TBL            = f"{CATALOG}.{SCHEMA}.dim_geo"
DIM_CAMPAIGN_TBL       = f"{CATALOG}.{SCHEMA}.dim_campaign"
DIM_SELLER_CHANNEL_TBL = f"{CATALOG}.{SCHEMA}.dim_seller_channel"
FACT_TBL               = f"{CATALOG}.{SCHEMA}.fact_marketing_daily"

KPI_BRAND_TBL          = f"{CATALOG}.{SCHEMA}.kpi_brand_performance"
KPI_CAMPAIGN_TBL       = f"{CATALOG}.{SCHEMA}.kpi_campaign_performance"
KPI_PLATFORM_TBL       = f"{CATALOG}.{SCHEMA}.kpi_platform_performance"
KPI_GEO_TBL            = f"{CATALOG}.{SCHEMA}.kpi_geo_performance"
KPI_TIER_TBL           = f"{CATALOG}.{SCHEMA}.kpi_product_tier_performance"
KPI_WATER_TBL          = f"{CATALOG}.{SCHEMA}.kpi_water_type_performance"
KPI_MONTHLY_TBL        = f"{CATALOG}.{SCHEMA}.kpi_monthly_performance"

tables = [
    ("DIM DATE", DIM_DATE_TBL),
    ("DIM PRODUCT", DIM_PRODUCT_TBL),
    ("DIM GEO", DIM_GEO_TBL),
    ("DIM CAMPAIGN", DIM_CAMPAIGN_TBL),
    ("DIM SELLER CHANNEL", DIM_SELLER_CHANNEL_TBL),
    ("FACT MARKETING DAILY", FACT_TBL),
    ("KPI BRAND", KPI_BRAND_TBL),
    ("KPI CAMPAIGN", KPI_CAMPAIGN_TBL),
    ("KPI PLATFORM", KPI_PLATFORM_TBL),
    ("KPI GEO", KPI_GEO_TBL),
    ("KPI PRODUCT TIER", KPI_TIER_TBL),
    ("KPI WATER TYPE", KPI_WATER_TBL),
    ("KPI MONTHLY", KPI_MONTHLY_TBL),
]

for title, tbl in tables:
    print("=" * 80)
    print(title)
    print("TABLE:", tbl)
    df = spark.table(tbl)
    print("ROWS:", df.count())
    display(df.limit(50))

DIM DATE
TABLE: workspace.water_bottle_db.dim_date
ROWS: 355


date_key,event_date,day,month,month_name,quarter,year,week_of_year,day_name,is_weekend
20241107,2024-11-07,7,11,November,4,2024,45,Thursday,false
20241009,2024-10-09,9,10,October,4,2024,41,Wednesday,false
20241017,2024-10-17,17,10,October,4,2024,42,Thursday,false
20241116,2024-11-16,16,11,November,4,2024,46,Saturday,true
20241016,2024-10-16,16,10,October,4,2024,42,Wednesday,false
20241206,2024-12-06,6,12,December,4,2024,49,Friday,false
20241202,2024-12-02,2,12,December,4,2024,49,Monday,false
20240716,2024-07-16,16,7,July,3,2024,29,Tuesday,false
20240630,2024-06-30,30,6,June,2,2024,26,Sunday,true
20240106,2024-01-06,6,1,January,1,2024,1,Saturday,true


DIM PRODUCT
TABLE: workspace.water_bottle_db.dim_product
ROWS: 418


product_key,brand,product_name,pack_size_ml
faf1354717b60d3ed4e30e1e4219bfc60f926a95541335b56f321b1ac192228e,HIMALAYAN,HIMALAYAN,200
0ed687e65b95479fc837d46bee40adc27b99c1e01488431f478f2cd47dc70b5f,KINLEY,KINLEY,200
ad2aab6f087b5aaf16e475c1e322495400fc05392447664716ef17273eccf2c2,BISLERI,BISLERI,1000
2a2f683986ae687135a30e4f452e3ac3719df3550912482450a2ceef311a968a,BISLERI,BISLERI,2000
5f31a333e553ea058dd54617627e84bba216fa17f6a80f2e88e2b8c294f1d1de,AQUAFINA,AQUAFINA,1000
0fb620c600a4908f767660ec709d60dfd073dda37090be68ac2afec05f10370d,AQUAFINA,AQUAFINA,2000
7290b5184facb08f54f2abff5089e8501c7fe8b212734d7a7e989104e9a07214,RAIL NEER,RAIL NEER,2000
5097ec67ecf1d888efbdb36f322f7864b1184c12ebd7c336a7b24a74a1dd7534,KINLEY,KINLEY,2000
282ba7fd0111d00047bd620131109b2d5713ed8e7d853943caf3785d86e85acc,BISLERI,BISLERI,1000
a1c9fa8ad07b0c5ef6b842d503a25932b12b38d82e7d8c8ba01110d05027e7bd,TATA COPPER+,TATA COPPER+,500


DIM GEO
TABLE: workspace.water_bottle_db.dim_geo
ROWS: 14


...

product_tier,brand,total_sales_units,total_revenue,total_profit,avg_marketing_roi,avg_roas
UNKNOWN,TATA COPPER+,705798,28968888.12,7379214.73,1.46602595,9.85566785
UNKNOWN,HIMALAYAN,678789,27801392.11,7200861.25,1.49840846,9.86656411
UNKNOWN,VEDICA,681090,28232628.79,7189965.17,1.46712193,9.76983435
UNKNOWN,RAIL NEER,687830,28099296.82,7231468.00,1.45857679,9.75217444
UNKNOWN,BISLERI,681446,28072447.08,7253782.29,1.50880041,9.91360382
UNKNOWN,BAILLEY,684851,28094637.14,7238146.56,1.50315699,9.88204919
UNKNOWN,KINLEY,706953,28663796.79,7309996.40,1.49304980,9.92729602
UNKNOWN,AQUAFINA,700762,29148299.71,7437835.90,1.47356022,9.87696206


KPI WATER TYPE
TABLE: workspace.water_bottle_db.kpi_water_type_performance
ROWS: 1


water_type,total_sales_units,total_revenue,total_profit,avg_marketing_roi,avg_roas
UNKNOWN,5527519,227081386.56,58241270.30,1.48349053,9.85573844


KPI MONTHLY
TABLE: workspace.water_bottle_db.kpi_monthly_performance
ROWS: 12


year,month,total_sales_units,total_revenue,total_profit,avg_marketing_roi,avg_roas
2024,3,488083,20079857.38,5157025.23,1.47394566,9.83934719
2024,12,268978,11106594.44,2852168.12,1.47724928,9.83695644
2024,8,483106,19742963.71,5070149.02,1.50404900,9.91286318
2024,7,473162,19624071.54,5002919.28,1.50428164,9.93241817
2024,6,482889,20000915.23,5144398.72,1.48867800,9.84950043
2024,5,487521,19959649.55,5096316.13,1.47095076,9.85506792
2024,2,454911,18635534.67,4798866.84,1.48839119,9.83466318
2024,11,470553,19380010.96,4939145.48,1.48024251,9.89352502
2024,4,467747,19164883.21,4919050.18,1.46044111,9.77529375
2024,1,491296,20153046.81,5150186.84,1.46554521,9.77929166


In [0]:
# ============================================
# GOLD OUTPUT SUMMARY CELL
# Row counts + schema + top sample
# ============================================

tables = [
    ("DIM DATE", "workspace.water_bottle_db.dim_date"),
    ("DIM PRODUCT", "workspace.water_bottle_db.dim_product"),
    ("DIM GEO", "workspace.water_bottle_db.dim_geo"),
    ("DIM CAMPAIGN", "workspace.water_bottle_db.dim_campaign"),
    ("DIM SELLER CHANNEL", "workspace.water_bottle_db.dim_seller_channel"),
    ("FACT MARKETING DAILY", "workspace.water_bottle_db.fact_marketing_daily"),
    ("KPI BRAND", "workspace.water_bottle_db.kpi_brand_performance"),
    ("KPI CAMPAIGN", "workspace.water_bottle_db.kpi_campaign_performance"),
    ("KPI PLATFORM", "workspace.water_bottle_db.kpi_platform_performance"),
    ("KPI GEO", "workspace.water_bottle_db.kpi_geo_performance"),
    ("KPI PRODUCT TIER", "workspace.water_bottle_db.kpi_product_tier_performance"),
    ("KPI WATER TYPE", "workspace.water_bottle_db.kpi_water_type_performance"),
    ("KPI MONTHLY", "workspace.water_bottle_db.kpi_monthly_performance"),
]

summary_rows = []
for title, tbl in tables:
    df = spark.table(tbl)
    summary_rows.append((title, tbl, df.count(), len(df.columns), ", ".join(df.columns[:8])))

summary_df = spark.createDataFrame(
    summary_rows,
    ["table_name", "table_path", "row_count", "column_count", "first_8_columns"]
)

display(summary_df)

table_name,table_path,row_count,column_count,first_8_columns
DIM DATE,workspace.water_bottle_db.dim_date,355,10,"date_key, event_date, day, month, month_name, quarter, year, week_of_year"
DIM PRODUCT,workspace.water_bottle_db.dim_product,418,4,"product_key, brand, product_name, pack_size_ml"
DIM GEO,workspace.water_bottle_db.dim_geo,14,4,"geo_key, region, state, city"
DIM CAMPAIGN,workspace.water_bottle_db.dim_campaign,26,5,"campaign_key, campaign_name, campaign_type, offer_type, promotion_flag"
DIM SELLER CHANNEL,workspace.water_bottle_db.dim_seller_channel,128,5,"seller_channel_key, seller, platform_source, channel, seller_type"
FACT MARKETING DAILY,workspace.water_bottle_db.fact_marketing_daily,47991,31,"fact_row_key, silver_ingestion_ts, event_date, date_key, product_key, geo_key, campaign_key, seller_channel_key"
KPI BRAND,workspace.water_bottle_db.kpi_brand_performance,8,6,"brand, total_sales_units, total_revenue, total_profit, avg_marketing_roi, avg_roas"
KPI CAMPAIGN,workspace.water_bottle_db.kpi_campaign_performance,24,8,"campaign_type, offer_type, promotion_flag, total_sales_units, total_campaign_revenue, avg_campaign_efficiency, avg_marketing_roi, avg_roas"
KPI PLATFORM,workspace.water_bottle_db.kpi_platform_performance,6,6,"platform_source, channel, total_sales_units, total_revenue, avg_rating, avg_roas"
KPI GEO,workspace.water_bottle_db.kpi_geo_performance,12,8,"region, state, city, total_sales_units, total_revenue, total_profit, avg_marketing_roi, avg_roas"


In [0]:
# ============================================
# GOLD BUSINESS VIEW CHECKS
# ============================================

print("FACT TABLE SAMPLE")
display(
    spark.table("workspace.water_bottle_db.fact_marketing_daily")
    .select(
        "event_date",
        "date_key",
        "product_key",
        "geo_key",
        "campaign_key",
        "seller_channel_key",
        "sales_units",
        "marketing_spend",
        "gross_revenue",
        "gross_profit",
        "marketing_roi",
        "roas"
    )
    .limit(50)
)

print("PRODUCT DIM SAMPLE")
display(
    spark.table("workspace.water_bottle_db.dim_product")
    .select("product_key", "brand", "product_name", "pack_size_ml")
    .limit(50)
)

print("CAMPAIGN KPI SAMPLE")
display(
    spark.table("workspace.water_bottle_db.kpi_campaign_performance")
    .limit(50)
)

print("MONTHLY KPI SAMPLE")
display(
    spark.table("workspace.water_bottle_db.kpi_monthly_performance")
    .orderBy("year", "month")
)

FACT TABLE SAMPLE


event_date,date_key,product_key,geo_key,campaign_key,seller_channel_key,sales_units,marketing_spend,gross_revenue,gross_profit,marketing_roi,roas
2024-11-19,20241119,f16804ad6334821fdcbf846ab5c3de6258c0ef4f942637197896f7e53cc9805a,d3690e606cf2a90a3d4ace81b5a4922b4c520ff8365bff07a663eb80f94c4eaa,6bdee57cca647245b6b63707dbbdd335e3b81529e7c94298ba47c4854cedfdd7,dbc0d4fa0ea9759824f63fc5fc34d5ac8eefcad17720d057687dcd20e51c25a0,191,2232.48,17587.28,4799.83,1.1500,7.8779
2024-10-06,20241006,898f173ccb7274f2434c703674e28e1cc90c357bd841f76eb75d10f53db16591,f5ed35492a84e9638f2164dd7f2e8142213b61e2b75b396ea8f171766cd211d0,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,7a03e650e89777d88242de56e1dd6d708b4e09a69dbed54b6fa7862c17bbae2b,147,498.87,5819.73,1562.61,2.1323,11.6658
2024-10-13,20241013,2c6fd7b138cc159510558f84d5622446b6df5439883f58c01db74a7572888ad2,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,13955b64a6bfa57bef2efc5d2224553cdfe435d157944fb4a62f20b82ed69662,100,191.37,2911.00,687.00,2.5899,15.2114
2024-11-23,20241123,bd0936c79a72013ab2f9460ce3e197532a38d42284e8fc92a1df6b1aa1b86f5e,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,1847218e97680115e7c7d774c1e3b2bdc41bcad73940987e473b162354d83d55,128,955.09,9891.84,2552.32,1.6723,10.3570
2024-10-30,20241030,ffdbd6dbbcce5df8c4767ade9be2b9ad24185cbe1134b69db6925dcce1e10031,6795197aeabac1a08fe0559a5f4715fb06230833591904fc71f6535c72cd4b4f,694a723154adab62867c2d41699bd498721b7999484255aad115a2b807847837,eea5f8632bf6ddc0a54ca39ebbf89f2621d0ff80ef58d3575758a2d31e3c89a2,190,607.13,3902.60,1409.80,1.3221,6.4279
2024-10-23,20241023,5b7fbf0d977c7ba06c641b15a98e965740141dffa31dcf5c313937ab09bdac41,d5ced0aecdedaf86f0eff2f38f96e0d4b759cc0985303bec3d16d02948740797,694a723154adab62867c2d41699bd498721b7999484255aad115a2b807847837,d3a60045372283cc1249afdd685de5083fd04cb0bc55ca7a4f5e400a9c818324,180,374.03,5941.80,2016.00,4.3899,15.8859
2024-11-14,20241114,bd0936c79a72013ab2f9460ce3e197532a38d42284e8fc92a1df6b1aa1b86f5e,7fe20ae1fce2c9c22af1816b3d8b7fd3b2ebc2e63f7944f8669bc36276120dd3,5cb03ed508e89f9f918a9c0f7ccbbb99cf4017040484c16b6ac123e4a865a99e,b99ce08ebc2b20833c2122fec734ff5517127460af08df94917ecdae821b1ec4,102,991.10,6114.90,1743.18,0.7588,6.1698
2024-11-07,20241107,5b7fbf0d977c7ba06c641b15a98e965740141dffa31dcf5c313937ab09bdac41,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,48145babe4c82539bc4ed6ab3251f484bf7d9c6e4b6567995ca13b03ac1152bc,a1b1b2d7226a504303b33f77a4222c8da46b9477fa89dfa7a0c53f8a4102e7e6,92,376.44,2934.80,642.16,0.7059,7.7962
2024-11-30,20241130,077d6ea902b464bab8ced7fb169da98373c651f3246cea4f8234c545b14678cb,62f5d1cc750bc2735fc2b5c95d7e4779af68258d13bc4dc4cad4b1a1d913d305,8561370ff68ffbc15db934a193cf0263cfeb4e9dc182f2f3e8006077bd2a3234,eea5f8632bf6ddc0a54ca39ebbf89f2621d0ff80ef58d3575758a2d31e3c89a2,81,122.67,1892.16,643.14,4.2428,15.4248
2024-10-15,20241015,4925ee45fb004629a81aeea660fc8a9ba3dd398698fad31fd359ffc6986c6ea3,6795197aeabac1a08fe0559a5f4715fb06230833591904fc71f6535c72cd4b4f,4babfa6e6f2eabe0be89a7cbb8cc1fe1cda1554759a0bb38983e4bf6b45f3ca2,a1f95ea0d32143b5cebe8a0f515e2708f92488e49bddef938aa9868995a04c45,150,1197.75,12034.50,3013.50,1.5160,10.0476


PRODUCT DIM SAMPLE


product_key,brand,product_name,pack_size_ml
faf1354717b60d3ed4e30e1e4219bfc60f926a95541335b56f321b1ac192228e,HIMALAYAN,HIMALAYAN,200
0ed687e65b95479fc837d46bee40adc27b99c1e01488431f478f2cd47dc70b5f,KINLEY,KINLEY,200
ad2aab6f087b5aaf16e475c1e322495400fc05392447664716ef17273eccf2c2,BISLERI,BISLERI,1000
2a2f683986ae687135a30e4f452e3ac3719df3550912482450a2ceef311a968a,BISLERI,BISLERI,2000
5f31a333e553ea058dd54617627e84bba216fa17f6a80f2e88e2b8c294f1d1de,AQUAFINA,AQUAFINA,1000
0fb620c600a4908f767660ec709d60dfd073dda37090be68ac2afec05f10370d,AQUAFINA,AQUAFINA,2000
7290b5184facb08f54f2abff5089e8501c7fe8b212734d7a7e989104e9a07214,RAIL NEER,RAIL NEER,2000
5097ec67ecf1d888efbdb36f322f7864b1184c12ebd7c336a7b24a74a1dd7534,KINLEY,KINLEY,2000
282ba7fd0111d00047bd620131109b2d5713ed8e7d853943caf3785d86e85acc,BISLERI,BISLERI,1000
a1c9fa8ad07b0c5ef6b842d503a25932b12b38d82e7d8c8ba01110d05027e7bd,TATA COPPER+,TATA COPPER+,500


CAMPAIGN KPI SAMPLE


campaign_type,offer_type,promotion_flag,total_sales_units,total_campaign_revenue,avg_campaign_efficiency,avg_marketing_roi,avg_roas
SEASONAL,BOGO,YES,171010,6908540.12,9.80461165,1.45830963,9.80461165
SEASONAL,NONE,NO,710835,0.00,9.83310852,1.47901948,9.83310852
ALWAYS ON,NONE,YES,156177,6565122.93,9.88629549,1.46710148,9.88629549
ALWAYS ON,BOGO,YES,156417,6476088.16,9.85149496,1.46660949,9.85149496
ALWAYS ON,DISCOUNT,YES,163186,6534082.18,9.87500584,1.50021431,9.87500584
FLASH SALE,NONE,YES,162607,6839539.01,9.96848456,1.50995649,9.96848456
FLASH SALE,DISCOUNT,YES,174010,7096433.83,9.86018442,1.46914895,9.86018442
ALWAYS ON,NONE,NO,724913,0.00,9.85161886,1.49024372,9.85161886
FESTIVE,NONE,NO,696543,0.00,9.78110420,1.46306610,9.78110420
FLASH SALE,BOGO,YES,169893,6977595.25,10.06235380,1.53546760,10.06235380


MONTHLY KPI SAMPLE


year,month,total_sales_units,total_revenue,total_profit,avg_marketing_roi,avg_roas
2024,1,491296,20153046.81,5150186.84,1.46554521,9.77929166
2024,2,454911,18635534.67,4798866.84,1.48839119,9.83466318
2024,3,488083,20079857.38,5157025.23,1.47394566,9.83934719
2024,4,467747,19164883.21,4919050.18,1.46044111,9.77529375
2024,5,487521,19959649.55,5096316.13,1.47095076,9.85506792
2024,6,482889,20000915.23,5144398.72,1.48867800,9.84950043
2024,7,473162,19624071.54,5002919.28,1.50428164,9.93241817
2024,8,483106,19742963.71,5070149.02,1.50404900,9.91286318
2024,9,475568,19696939.27,5143343.80,1.50062348,9.82694493
2024,10,483705,19536919.79,4967700.66,1.48499081,9.92249301
